# Proyecto de recuperación de la información
## Construcción del índice
A continuación descargamos la data que se encuentra en Huggingfaces con el corpus llamado fiqa

Instalamos la librería de datasets de Huggingface y descargamos el corpus fiqa

In [8]:
%pip install datasets
%pip install nltk


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Importamos la librería datasets y cargamos el corpus fiqa, luego imprimimos el número de documentos, las columnas y el primer documento del corpus.

In [5]:
from datasets import load_dataset
corpus = load_dataset("BeIR/fiqa", "corpus", split = "corpus")
print("Número de documentos:", len(corpus))
print("Columnas", corpus.column_names)
documento = corpus[0]
print("ID\n", documento["_id"])
print("Título\n", documento["title"])
print("Texto\n", documento["text"])





Número de documentos: 57638
Columnas ['_id', 'title', 'text']
ID
 3
Título
 
Texto
 I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange for their massive investment in education, rather than getting out with thousands in student debt and then complaining that they aren't qualified to do anything.


Realizamos el procesamiento básico que incluye tokenización, normalización y remoción de stopwords. Para esto utilizamos la librería nltk y descargamos el conjunto de stopwords en español.

In [13]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
def tokenizar(texto):
    return re.findall(r"[a-z]+|[0-9]+(?:\.[0-9]+)?", texto.lower())
stopwords_ingles = set(stopwords.words("english"))
def quitar_stopwords(tokens):
    return[token for token in tokens if token not in stopwords_ingles]
tokens = tokenizar(corpus[0]["text"])
tokens_filtrados = quitar_stopwords(tokens)
print("Palabras en la lista de NLTK:", len(stopwords_ingles))
print("Antes:", tokens[:20])
print("Después:", tokens_filtrados[:20])



Palabras en la lista de NLTK: 198
Antes: ['i', 'm', 'not', 'saying', 'i', 'don', 't', 'like', 'the', 'idea', 'of', 'on', 'the', 'job', 'training', 'too', 'but', 'you', 'can', 't']
Después: ['saying', 'like', 'idea', 'job', 'training', 'expect', 'company', 'training', 'workers', 'job', 'building', 'software', 'perhaps', 'educational', 'systems', 'u', 'students', 'worry', 'little', 'getting']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [20]:
from collections import defaultdict
def construir_indice(documentos):
    indice = defaultdict(set)
    for doc in documentos:
        tokens = tokenizar(doc["text"])
        terminos = quitar_stopwords(tokens)
        for termino in set(terminos):
            indice[termino].add(doc["_id"])
    return indice
indice = construir_indice(corpus)
print("Indice construido:", len(indice))
print("Primeros 10 términos del índice:", list(indice.keys())[:10])


Indice construido: 74784
Primeros 10 términos del índice: ['idea', 'massive', 'like', 'software', 'systems', 'building', 'education', 'getting', 'perhaps', 'skills']


### Búsqueda basada en similitud Jaccard utilizando vectores binarios

In [28]:
from collections import Counter
tamanos = Counter()
for id_documentos in indice.values():
    tamanos.update(id_documentos)
def buscar_jaccard(consulta,indice,tamanos, k=10):
    terminos_consulta = set(quitar_stopwords(tokenizar(consulta)))
    coincidencias = Counter()
    for termino in terminos_consulta:
        coincidencias.update(indice.get(termino, ()))
    resultados = []
    for id_documento, interseccion in coincidencias.items():
        union = tamanos[id_documento] + len(terminos_consulta) - interseccion
        jaccard = interseccion / union
        resultados.append((id_documento, jaccard))
    resultados.sort(key=lambda x: x[1], reverse=True)
    return resultados[:k]



In [29]:
documentos_prueba = [
    {"_id": "d1", "text": "credit"},
    {"_id": "d2", "text": "credit card loan bank"},
    {"_id": "d3", "text": "mortgage"},
]

indice_prueba = construir_indice(documentos_prueba)

tamanos_prueba = Counter()
for ids_documentos in indice_prueba.values():
    tamanos_prueba.update(ids_documentos)

resultados = buscar_jaccard("credit card", indice_prueba, tamanos_prueba)

# d1 obtiene 1/2 y d2 obtiene 2/4. d3 se omite porque no comparte términos.
assert dict(resultados) == {"d1": 0.5, "d2": 0.5}, resultados

resultados_repetidos = buscar_jaccard(
    "credit credit card", indice_prueba, tamanos_prueba
)
assert dict(resultados_repetidos) == dict(resultados), resultados_repetidos

print("Prueba de Jaccard aprobada")

Prueba de Jaccard aprobada


### TF-IDF

In [30]:
from math import log,sqrt
def construir_indice_tfidf(documentos):
    frecuencias = defaultdict(dict)

    for documento in documentos:
        doc_id = documento["_id"]
        tokens = quitar_stopwords(tokenizar(documento["text"]))

        for termino, frecuencia in Counter(tokens).items():
            frecuencias[termino][doc_id] = frecuencia

    n_documentos = len(documentos)
    idf = {}
    cuadrados = defaultdict(float)

    for termino, pesos_documentos in frecuencias.items():
        idf[termino] = log((n_documentos + 1) / (len(pesos_documentos) + 1)) + 1

        for doc_id, frecuencia in pesos_documentos.items():
            peso = frecuencia * idf[termino]
            pesos_documentos[doc_id] = peso
            cuadrados[doc_id] += peso ** 2

    normas = {doc_id: sqrt(total) for doc_id, total in cuadrados.items()}
    return frecuencias, idf, normas
def buscar_coseno(consulta, indice_tfidf, idf, normas, k=10):
    tokens = quitar_stopwords(tokenizar(consulta))
    pesos_consulta = {
        termino: frecuencia * idf[termino]
        for termino, frecuencia in Counter(tokens).items()
        if termino in idf
    }

    if not pesos_consulta:
        return []

    norma_consulta = sqrt(sum(peso ** 2 for peso in pesos_consulta.values()))
    productos = defaultdict(float)

    for termino, peso_consulta in pesos_consulta.items():
        for doc_id, peso_documento in indice_tfidf[termino].items():
            productos[doc_id] += peso_consulta * peso_documento

    resultados = [
        (doc_id, producto / (norma_consulta * normas[doc_id]))
        for doc_id, producto in productos.items()
    ]
    resultados.sort(key=lambda resultado: (-resultado[1], resultado[0]))
    return resultados[:k]

### BM25
